# Snake-RepairLLaMA — LoRA Training on CodeLlama-7b-hf

**What this trains:** a LoRA adapter on top of `codellama/CodeLlama-7b-hf` (the base FIM-supporting variant) using the IR4/OR2 dataset (train: 80,597 / val: 8,956).

**Why this base instead of `-Python-hf`:** the eval pipeline (notebooks 01–04) uses `<FILL_ME>` IR4 prompts. `<FILL_ME>` expands into FIM token IDs 32007/32008/32009 which don't exist in Python-hf's 32000-row vocab. The base 7b-hf has 32016 vocab including those rows. **Same prompt format end-to-end.**

**Runtime:** ~60–90 min on RunPod A100 80GB. ~3 hours on RTX 4090. Cost: ~$1.50–2.00.

**Output:** LoRA adapter pushed to HF Hub. Use the repo path in `notebooks/04_run1_snakellama.ipynb` as `ADAPTER_NAME` for eval.

## Step-by-step
1. Setup + deps + GPU check
2. Configuration (one cell — edit and re-run)
3. Load + peek at training data
4. Tokenizer + verify FIM tokens
5. Tokenize datasets (mask input tokens in labels — completion-only training)
6. Load base in 4-bit, attach LoRA, print trainable param count
7. TrainingArguments with `eval_steps` so val loss is visible mid-run
8. Train (live loss + val loss in output)
9. Save + push adapter to HF Hub
10. Sanity-check inference on 1 QuixBugs bug


## 1. Setup — clone repo & install deps

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/AliSuleman27/snake-repairllama-baseline.git"

# Detect environment and clone if needed
if os.path.isdir("/workspace") and not os.path.isdir("/workspace/snake-repairllama-baseline"):
    subprocess.run(["git", "clone", REPO_URL, "/workspace/snake-repairllama-baseline"], check=True)
    os.chdir("/workspace/snake-repairllama-baseline")
elif os.path.isdir("/workspace/snake-repairllama-baseline"):
    os.chdir("/workspace/snake-repairllama-baseline")
    subprocess.run(["git", "-C", os.getcwd(), "pull"], check=False)
elif os.path.isdir("/content"):
    if not os.path.isdir("/content/snake-repairllama-baseline"):
        subprocess.run(["git", "clone", REPO_URL, "/content/snake-repairllama-baseline"], check=True)
    os.chdir("/content/snake-repairllama-baseline")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("cwd:", os.getcwd())
print("repo contents:", sorted(os.listdir(".")))


Already up to date.
cwd: /workspace/snake-repairllama-baseline
repo contents: ['.cfg', '.git', '.gitignore', '.ipynb_checkpoints', 'README.md', 'codellama models comparision.png', 'data', 'notebooks', 'other model benchmarking results.png', 'requirements.txt', 'results', 'scripts', 'src', 'train']


In [2]:
# Pinned versions - known-good combo. Avoid the FP8/MoE schema regression in newer transformers.
# bitsandbytes>=0.45 is required for Blackwell GPUs (RTX 50xx / RTX PRO 4500 / 6000).
# Older 0.44.x works on Ada/Ampere/Hopper but errors on import for Blackwell.
%pip install -U "transformers==4.46.3" "peft==0.13.2" "accelerate==1.0.1" "bitsandbytes>=0.45" "datasets==3.0.1" "huggingface_hub<0.26" tensorboard sentencepiece


INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 207.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 497.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 463.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 289.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 323.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 473.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 439.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.8/799.8 kB 180.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 420.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 216.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 480.6

In [3]:
import os, json, gc, torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}")
    print(f"VRAM: {p.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected. Training requires a CUDA GPU.")


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


## 2. HuggingFace login

CodeLlama is gated on HF — you need a token with **read** access to the model and **write** access (so you can push the trained adapter back). Either:
- Set `HF_TOKEN` env var / Colab secret, or
- Paste it at the prompt below.


In [4]:
token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if not token:
    from getpass import getpass
    token = getpass("Paste HF token (write access — needed to push adapter): ")

from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("HF login OK")


Paste HF token (write access — needed to push adapter):  ········


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful
HF login OK


## 3. Configuration — edit this cell

These values control the entire run. Defaults are tuned for an A100 80GB. See the README for batch sizes on smaller GPUs.


In [ ]:
# # === Model ===
# BASE_MODEL = "codellama/CodeLlama-7b-hf"

# # === Where to save the trained adapter ===
# HF_USERNAME      = "alisuleman525"   # <-- EDIT THIS to your HF username
# HF_ADAPTER_REPO  = f"{HF_USERNAME}/snake-repairllama-7b-fim-r16"
# LOCAL_OUTPUT_DIR = "./train/output/snake-repairllama-7b-fim-r16"

# # === Data paths (relative to repo root) ===
# TRAIN_PARQUET = "train/data/train.parquet"
# VAL_PARQUET   = "train/data/validation.parquet"

# # === LoRA config ===
# LORA_R              = 16
# LORA_ALPHA          = 32
# LORA_DROPOUT        = 0.05
# LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
#                        "gate_proj", "up_proj", "down_proj"]
# # All linear layers — better fine-tune quality, ~30M trainable params (~0.5% of 7B).
# # Ahsan used just q_proj/v_proj which is ~4M params — much less expressive.

# # === Training hyperparameters ===
# NUM_EPOCHS              = 1     # 1 epoch on 80k is usually enough; bump to 2 if val loss still falling
# LEARNING_RATE           = 2e-4  # Standard QLoRA value. Ahsan used 5e-4 (high)
# PER_DEVICE_BATCH_SIZE   = 8     # A100 80GB: 8 (16 OOMs at seq 1024 + no checkpoint). A100 40GB: 4. A6000/4090: 4.
# GRAD_ACCUM              = 4     # A100 80GB: 4 (effective batch=32). A100 40GB: 8. Smaller GPU: 8.
# MAX_SEQ_LENGTH          = 1024  # Ahsan used 512; we go higher to fit longer eval prompts
# WARMUP_STEPS            = 100
# LR_SCHEDULER_TYPE       = "cosine"

# # === Eval / save / log frequencies ===
# EVAL_STEPS         = 200    # see val loss every ~5 min on A100
# SAVE_STEPS         = 400   # must be a multiple of EVAL_STEPS for load_best_model_at_end
# LOGGING_STEPS      = 50
# SAVE_TOTAL_LIMIT   = 2

# # === Quantization ===
# USE_4BIT = True   # 4-bit NF4 (QLoRA) — best memory/quality tradeoff for 7B

# print(f"Base model:       {BASE_MODEL}")
# print(f"Adapter target:   {HF_ADAPTER_REPO}")
# print(f"Local output:     {LOCAL_OUTPUT_DIR}")
# print(f"Effective batch:  {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM}")
# print(f"LoRA target:      {LORA_TARGET_MODULES}")


In [34]:
# === Match RepairLLaMA paper exactly ===
BASE_MODEL          = "codellama/CodeLlama-7b-hf"

HF_USERNAME      = "alisuleman525"   # <-- EDIT THIS to your HF username
HF_ADAPTER_REPO  = f"{HF_USERNAME}/snake-repairllama-7b-fim-run3"
LOCAL_OUTPUT_DIR = "./train/output/snake-repairllama-7b-fim-run3"

TRAIN_PARQUET = "train/data/train.parquet"
VAL_PARQUET   = "train/data/validation.parquet"

LORA_R              = 8                     # was 16
LORA_ALPHA          = 16                    # was 32
LORA_DROPOUT        = 0.05                  # same
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # was all 7 linear

NUM_EPOCHS          = 1                     # was 1
LEARNING_RATE       = 5e-4                  # was 2e-4
LR_SCHEDULER_TYPE   = "cosine"              # same
MAX_SEQ_LENGTH      = 1024                  # same
WARMUP_STEPS        = 100                   # paper doesn't specify; 100 is fine

# Effective batch = 64 (their 16×4)
PER_DEVICE_BATCH_SIZE = 8                  # paper used 16 per GPU
GRAD_ACCUM            = 8                   # 16 × 4 = 64 on a single GPU

# Eval/save unchanged
EVAL_STEPS         = 200
SAVE_STEPS         = 400
LOGGING_STEPS      = 50
SAVE_TOTAL_LIMIT   = 2
USE_4BIT           = True

print(f"Base model:       {BASE_MODEL}")
print(f"Adapter target:   {HF_ADAPTER_REPO}")
print(f"Local output:     {LOCAL_OUTPUT_DIR}")
print(f"Effective batch:  {PER_DEVICE_BATCH_SIZE * GRAD_ACCUM}")
print(f"LoRA target:      {LORA_TARGET_MODULES}")

Base model:       codellama/CodeLlama-7b-hf
Adapter target:   alisuleman525/snake-repairllama-7b-fim-run3
Local output:     ./train/output/snake-repairllama-7b-fim-run3
Effective batch:  64
LoRA target:      ['q_proj', 'v_proj']


## 4. Load + peek at the training data

The dataset has two fields per row:
- **`input`** (IR4) — buggy function with commented buggy lines + `<FILL_ME>` placeholder
- **`output`** (OR2) — the fix that goes where `<FILL_ME>` is

Stratified mix of three sources:
- RunBugRun (~78%)
- RepairLLaMA dataset (~19%)
- PyResBugs (~3%)


In [6]:
ds_train = load_dataset("parquet", data_files=TRAIN_PARQUET, split="train")
ds_val   = load_dataset("parquet", data_files=VAL_PARQUET,   split="train")
print(f"Train: {len(ds_train):,} rows")
print(f"Val:   {len(ds_val):,} rows")
print(f"Columns: {ds_train.column_names}")


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 80,597 rows
Val:   8,956 rows
Columns: ['input', 'output']


In [7]:
ex = ds_train[0]
print("=== INPUT (IR4) ===")
print(ex["input"])
print()
print("=== OUTPUT (OR2) ===")
print(ex["output"])
print()
print(f"Has <FILL_ME>: {'<FILL_ME>' in ex['input']}")


=== INPUT (IR4) ===
# -*- coding: utf-8 -*-

def main():
    _, K = map(int, input().split(' '))
    S = input()

# Buggy code:
#     S[K - 1] = S[K - 1].lower()
# 
#     print(S)
<FILL_ME>


if __name__ == '__main__':
    main()


=== OUTPUT (OR2) ===
    print(S[:K - 1] + S[K - 1].lower() + S[K:])


Has <FILL_ME>: True


## 5. Tokenizer — verify FIM expansion works

CodeLlama's fast tokenizer auto-expands `<FILL_ME>` into the FIM sequence `<PRE> prefix <SUF> suffix <MID>`. We need to verify this works on `7b-hf` (vocab=32016, includes FIM rows).


In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

sample = ("def foo(x):\n"
          "    if x > 0:\n"
          "# Buggy:\n"
          "#         return -x\n"
          "<FILL_ME>\n"
          "    return 0\n")
ids = tokenizer(sample, add_special_tokens=True).input_ids
print(f"vocab_size: {tokenizer.vocab_size}")
print(f"max token id in sample: {max(ids)}")
print(f"FIM tokens present (32007/32008/32009): {any(t in ids for t in [32007, 32008, 32009])}")
print(f"Decoded sample: {tokenizer.decode(ids)[:200]}")
assert max(ids) < tokenizer.vocab_size, (
    "Out-of-vocab tokens detected — wrong base? "
    "Make sure BASE_MODEL is 'codellama/CodeLlama-7b-hf', NOT '-Python-hf'."
)
print("✓ Tokenizer FIM works")


tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab_size: 32016
max token id in sample: 32009
FIM tokens present (32007/32008/32009): True
Decoded sample: <s> <PRE> def foo(x):
    if x > 0:
# Buggy:
#         return -x
 <SUF>
    return 0
 <MID>
✓ Tokenizer FIM works


## 6. Tokenize for completion-only training

For each `(input, output)` pair the model learns to predict `output` *given* `input`. We do this by:

1. Tokenize input → `<s> <PRE> prefix <SUF> suffix <MID>`
2. Tokenize output → `output_tokens`
3. Concatenate: `full = input_tokens + output_tokens + [eos]`
4. Build labels: same as `full` but with **-100 on the input portion** so loss is computed only on the output tokens.

This is the standard "completion-only" pattern. Without label masking the model would also learn to "predict" the input, which is wasteful and noisy.

If the full sequence exceeds `MAX_SEQ_LENGTH`, we truncate from the **left** (drop the early prefix) so the output tokens are always preserved.


In [9]:
def tokenize_example(example):
    input_ids_in  = tokenizer(example["input"],  add_special_tokens=True).input_ids
    input_ids_out = tokenizer(example["output"], add_special_tokens=False).input_ids
    eos = tokenizer.eos_token_id

    full_ids = input_ids_in + input_ids_out + [eos]
    labels   = [-100] * len(input_ids_in) + input_ids_out + [eos]

    if len(full_ids) > MAX_SEQ_LENGTH:
        excess = len(full_ids) - MAX_SEQ_LENGTH
        full_ids = full_ids[excess:]
        labels   = labels[excess:]

    return {
        "input_ids": full_ids,
        "labels":    labels,
        "attention_mask": [1] * len(full_ids),
    }

print("Tokenizing train ...")
ds_train_tok = ds_train.map(tokenize_example, remove_columns=ds_train.column_names, num_proc=4, desc="train")
print("Tokenizing val ...")
ds_val_tok   = ds_val.map(tokenize_example,   remove_columns=ds_val.column_names,   num_proc=4, desc="val")

lengths = [len(x) for x in ds_train_tok["input_ids"][:2000]]
print(f"\nTrain length stats (n=2000 sample):")
print(f"  min={min(lengths)}  median={int(np.median(lengths))}  max={max(lengths)}  mean={int(np.mean(lengths))}")
print(f"  fraction at MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l == MAX_SEQ_LENGTH) / len(lengths):.1%}")


Tokenizing train ...


train (num_proc=4):   0%|          | 0/80597 [00:00<?, ? examples/s]

Tokenizing val ...


val (num_proc=4):   0%|          | 0/8956 [00:00<?, ? examples/s]


Train length stats (n=2000 sample):
  min=84  median=205  max=1002  mean=245
  fraction at MAX_SEQ_LENGTH=1024: 0.0%


In [10]:
ex = ds_train_tok[0]
print(f"input_ids ({len(ex['input_ids'])} tokens):")
print(ex["input_ids"][:30], "...")
print()
print("Decoded full sequence:")
print(tokenizer.decode(ex["input_ids"])[:500])
print()
masked = sum(1 for l in ex["labels"] if l == -100)
print(f"Label masking:")
print(f"  {masked} tokens masked (input prefix — no loss)")
print(f"  {len(ex['labels']) - masked} tokens trained on (output)")


input_ids (126 tokens):
[1, 32007, 396, 448, 29930, 29899, 14137, 29901, 23616, 29899, 29947, 448, 29930, 29899, 13, 13, 1753, 1667, 7295, 13, 1678, 17117, 476, 353, 2910, 29898, 524, 29892, 1881, 2141] ...

Decoded full sequence:
<s> <PRE> # -*- coding: utf-8 -*-

def main():
    _, K = map(int, input().split(' '))
    S = input()

# Buggy code:
#     S[K - 1] = S[K - 1].lower()
# 
#     print(S)
 <SUF>


if __name__ == '__main__':
    main()
 <MID>     print(S[:K - 1] + S[K - 1].lower() + S[K:])
</s>

Label masking:
  98 tokens masked (input prefix — no loss)
  28 tokens trained on (output)


## 7. Load the base model in 4-bit (QLoRA)

4-bit NF4 quantization shrinks the 7B base from ~13 GB (fp16) to ~3.5 GB. ~5% slower than fp16 but lets us use bigger batches and fit on smaller GPUs. This is the standard QLoRA recipe (Dettmers et al. 2023).


In [33]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if USE_4BIT else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
# Belt-and-suspenders: force checkpointing OFF in case the prepare flag is ignored
try:
    model.gradient_checkpointing_disable()
except Exception:
    pass
model.config.use_cache = False  # standard for training
print(f"Base model loaded. VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded. VRAM: 8.91 GB


In [35]:
import torch
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.1f} GB")

Allocated: 8.9 GB
Reserved: 10.0 GB


In [36]:
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

VRAM: 8.91 GB


In [37]:
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=LORA_TARGET_MODULES,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


trainable params: 4,194,304 || all params: 6,742,740,992 || trainable%: 0.0622


## 8. Training arguments + Trainer

Trainer prints loss every `LOGGING_STEPS` (50) and runs validation every `EVAL_STEPS` (200). What to watch:

- **Train loss decreasing**: expected, fine.
- **Val loss decreasing then plateauing/rising**: ideal. The rise is overfitting onset.
- **Val loss flat from the start**: data/format mismatch, LR wrong, or something else broken — abort, share the output, we'll diagnose.
- **Train loss = NaN**: LR too high or fp16 instability — lower LR, or set `fp16=False, bf16=True` if your GPU supports bf16 (A100 does).

Estimated steps for 1 epoch on 80k examples at effective batch 32: **2,520 steps**.


In [38]:
training_args = TrainingArguments(
    output_dir=LOCAL_OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=False,   # A100 has VRAM; checkpointing costs ~30 percent speed
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_steps=WARMUP_STEPS,
    optim="paged_adamw_8bit",
    bf16=True,                     # A100 native bf16 - better numerics than fp16
    fp16=False,
    tf32=True,                     # free matmul speedup on Ampere/A100

    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    logging_steps=LOGGING_STEPS,
    logging_first_step=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="tensorboard",
    push_to_hub=False,  # we push manually after training to control naming

    dataloader_pin_memory=True,
    dataloader_num_workers=8,      # A100 pods have 8-16 vCPUs - keep GPU fed
)

steps_per_epoch = len(ds_train_tok) // (PER_DEVICE_BATCH_SIZE * GRAD_ACCUM)
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps: {steps_per_epoch * NUM_EPOCHS}")


Steps per epoch: 1259
Total steps: 1259


In [39]:
# Ensure right-padding (left would shift loss positions for causal LM training)
tokenizer.padding_side = "right"

# DataCollatorForSeq2Seq pads input_ids and labels independently —
# the right choice for completion-only causal LM training (despite the Seq2Seq name).
# DataCollatorForLanguageModeling does NOT pad a separate labels field.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_tok,
    eval_dataset=ds_val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
)


## 9. Train

This is the long-running cell. Output streams: every 50 steps you see `loss=`, every 200 steps you see `eval_loss=`. If you're on a flaky network connection, run inside `tmux` on RunPod or use Colab background execution.


In [40]:
trainer.train()

Step,Training Loss,Validation Loss
200,0.224200,0.234315
400,0.237100,0.233889
600,0.231800,0.230858
800,0.227600,0.230888
1000,0.222900,0.228861
1200,0.240900,0.230458


TrainOutput(global_step=1259, training_loss=0.24770259544714943, metrics={'train_runtime': 15556.3248, 'train_samples_per_second': 5.181, 'train_steps_per_second': 0.081, 'total_flos': 1.885514210858238e+18, 'train_loss': 0.24770259544714943, 'epoch': 0.9997022332506204})

## 10. Save + push to HF Hub

`trainer.save_model()` writes the adapter (just the LoRA weights, not the full base) to `LOCAL_OUTPUT_DIR`. We then push to HF Hub as a private repo so it's easy to load from inference notebooks.


In [41]:
trainer.save_model(LOCAL_OUTPUT_DIR)
tokenizer.save_pretrained(LOCAL_OUTPUT_DIR)
print(f"Saved locally: {LOCAL_OUTPUT_DIR}")

trainer.model.push_to_hub(HF_ADAPTER_REPO, private=True)
tokenizer.push_to_hub(HF_ADAPTER_REPO, private=True)
print(f"Pushed to HF Hub: https://huggingface.co/{HF_ADAPTER_REPO}")


Saved locally: ./train/output/snake-repairllama-7b-fim-run3


adapter_model.safetensors:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

Pushed to HF Hub: https://huggingface.co/alisuleman525/snake-repairllama-7b-fim-run3


## 11. Sanity-check inference

Generate one patch on the first QuixBugs bug. The trained adapter should produce something that **looks like Python** and **attempts to fix the bug** (`n &= n - 1` for the bitcount example, or a close variant). If the output is gibberish or empty, the training didn't take — share the output and we'll diagnose.


In [45]:
# del trainer
gc.collect(); torch.cuda.empty_cache()

model.eval()

with open("data/quixbugs_eval.jsonl") as f:
    bug = json.loads(f.readline())

print("=== INPUT (IR4) ===")
print(bug["input"])
print()
print("=== GOLD (OR2) ===")
print(bug["output"])
print()

inputs = tokenizer(bug["input"], return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=1.0,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )
gen = tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== TRAINED ADAPTER OUTPUT ===")
print(gen)


=== INPUT (IR4) ===
def bitcount(n):
    count = 0
    while n:
# Buggy code:
#         n ^= n - 1
<FILL_ME>
        count += 1
    return count


=== GOLD (OR2) ===
        n &= n - 1


=== TRAINED ADAPTER OUTPUT ===
        n &= n - 1



## What's next

1. **Run `notebooks/04_run1_snakellama.ipynb`** with `ADAPTER_NAME = HF_ADAPTER_REPO` to get full QuixBugs + BugsInPy metrics.
2. **Compare to baseline** — `results/quixbugs_codellama_baseline.jsonl` and `results/bugsinpy_codellama_baseline.jsonl` are your reference numbers.
3. **TERMINATE THE POD** to stop billing if you're on RunPod.

If results look weak (val loss flat, sanity check garbled, low metrics), the most common fixes:
- Increase `NUM_EPOCHS` to 2 (cheap iteration).
- Lower `LEARNING_RATE` to `1e-4` if loss was unstable.
- Verify the sanity check output before booking another full eval run.
